<a href="https://colab.research.google.com/github/jameschen2004/FunderWonder/blob/main/FunderWonder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FunderWonder

---
## Setup

Run the cell below to install the required packages. You may see some warnings or dependency messages; these are safe to ignore.

In [ ]:
 # Install required packages (this may take a minute, and you may ignore the errors)
!pip install -qU langchain-google-genai
!pip -q install google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.0 MB/s eta 0:00:00


In [ ]:
# Configure your API keys
# You should already have GOOGLE_API_KEY saved in your Colab secrets.

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GCP_CREDENTIALS"] = userdata.get("GCP_CREDENTIALS")
print("API keys configured successfully!")

API keys configured successfully!


### Step 1: Create the LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

---
## Part A: Implement Grants.Gov API Tool


In [ ]:
import requests
from langchain.tools import tool

# Common headers to avoid being blocked by government APIs
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

@tool
def search_grants(keywords: str) -> str:
    """
    Searches Grants.gov for open and forecasted grants.
    Use this to find a list of potential grant opportunities.
    """
    url = "https://api.grants.gov/v1/api/search2"

    payload = {
        "keyword": keywords,
        "oppStatuses": "posted|forecasted",
        "rows": 10
    }

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            results = response.json()

            # Extract opportunities from nested structure
            data = results.get("data", {})
            opportunities = data.get("oppHits") or []

            output = ""
            for opp in opportunities:
                output += f"ID: {opp.get('id')} | Number: {opp.get('number')} | Title: {opp.get('title')}\n"

            return output if output else "No grants found for these keywords."
        else:
            return f"Error: Received status code {response.status_code}"
    except Exception as e:
        return f"Error searching grants: {str(e)}"


@tool
def get_grant_details(opportunity_id: str) -> str:
    """
    Fetches the full description and synopsis for a specific grant ID.
    Use this to see if a grant covers specific costs like travel or equipment.
    """
    url = "https://api.grants.gov/v1/api/fetchOpportunity"
    payload = {"opportunityId": opportunity_id}

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            data = response.json()

            # 1. Check for Synopsis (Posted Grants)
            synopsis = data.get("synopsis")
            if synopsis:
                # Try explanation first, then general description
                desc = synopsis.get("synopsisExplanation") or synopsis.get("synopsisDesc")
                if desc:
                    return desc

            # 2. Check for Forecast (Forecasted Grants)
            forecast = data.get("forecast")
            if forecast:
                desc = forecast.get("forecastExplanation") or forecast.get("forecastDesc")
                if desc:
                    return f"[Forecasted Grant] {desc}"

            return "No detailed description available in synopsis or forecast sections."
        else:
            return f"Error: Could not fetch details for ID {opportunity_id}."
    except Exception as e:
        return f"Error fetching details: {str(e)}"

In [ ]:
agent_prompt = """You are FunderWonder, a helpful research assistant seeking funding opportunities.

Your Goal:
Help researchers find grants that match their needs using the Grants.gov API.

Tools available:
1. search_grants(keywords): Searches for grants. ALWAYS use this first when a user asks for funding.
2. get_grant_details(opportunity_id): Gets full details for a specific grant ID.

Process:
- When a user asks for grants on a topic, immediately call `search_grants` with relevant keywords.
- Do not ask clarifying questions before searching unless the request is completely empty.
- Once you get a list of grants, summarize them for the user.
- If the user asks for more details on a specific one, use `get_grant_details`.
"""

grant_agent = create_agent(
    model=llm,
    tools=[search_grants, get_grant_details],
    system_prompt=agent_prompt
)

In [ ]:
test_query = "Find cancer research grants and summarize them"

for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': '6b0dd102-1a21-42bb-8f6a-7c575b49c315', 'name': 'search_grants', 'args': {'keywords': 'cancer research'}}]

Step: tools
Content: [{'type': 'text', 'text': 'ID: 360918 | Number: PAR-25-444 | Title: Cancer Center Support Grants (CCSGs) for NCI-designated Cancer Centers (P30 Clinical Trial Optional)\nID: 359855 | Number: FOR-CA-25-074 | Title: Forecast to Publish a Funding Opportunity Announcement for Advanced Development of Informatics Technologies for Cancer Research and Management (U24 Clinical Trial Optional)\nID: 359853 | Number: FOR-CA-25-075 | Title: Forecast to Publish a Funding Opportunity Announcement for Sustained Support for Informatics Technologies for Cancer Research and Management (U24 Clinical Trial Optional)\nID: 359892 | Number: FOR-CA-25-073 | Title: Forecast to Publish a Funding Opportunity Announcement for Early-Stage Development of Informatics Technologies for Cancer Research and Management (U01 Clinical Trial Optiona

---
## Part B: Authorize Google Drive/Google Docs

In [ ]:
import json
from google_auth_oauthlib.flow import Flow

secret_value = userdata.get('GCP_CREDENTIALS')

# Write to a temporary local file so the Google library can read it
with open('credentials.json', 'w') as f:
    f.write(secret_value)

SCOPES = [
    'https://www.googleapis.com/auth/documents',
    'https://www.googleapis.com/auth/drive.file'
]

flow = Flow.from_client_secrets_file(
    'credentials.json',
    scopes=SCOPES,
    redirect_uri='urn:ietf:wg:oauth:2.0:oob' # URI helps with readability
)

# Get the URL for you to visit
auth_url, _ = flow.authorization_url(prompt='consent')

print(f"1. Visit this URL to authorize: {auth_url}")
print("\n2. Sign in and click 'Allow'.")
print("3. You will see a page with a code. Copy it.")

code = input("\n4. Enter the authorization code here: ")

# Exchange the code for the actual credentials
flow.fetch_token(code=code)
creds = flow.credentials

print("\nAuthentication successful! Your agent is ready to write to Google Docs.")

1. Visit this URL to authorize: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=584926836345-rt23bmn5p3o2jamehuol812qameqn9bn.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocuments+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=Y1r6DjWZ5LqIE8PfnPS9vNHgs3bj1B&prompt=consent&access_type=offline

2. Sign in and click 'Allow'.
3. You will see a page with a code. Copy it.

4. Enter the authorization code here: 4/1AfrIepChiNBqNUqsah0iLCvfHByyUXWH4TSQ6AWX4Av0XL5bAoMrdeB4gXU

Authentication successful! Your agent is ready to write to Google Docs.


---
## Part C: Implement Google Docs API Tool

In [ ]:
from googleapiclient.discovery import build

# Build the Docs service using your active credentials
docs_service = build('docs', 'v1', credentials=creds)

@tool
def create_proposal_doc(title: str, content: str) -> str:
    """
    Creates a new Google Doc with the specified title and writes the content into it.
    Returns the URL of the created document.
    """
    try:
        # Step 1: Create a blank document
        doc = docs_service.documents().create(body={'title': title}).execute()
        doc_id = doc.get('documentId')

        # Step 2: Insert the text into the document
        # We use index 1 because index 0 is the start of the document structure
        requests = [
            {
                'insertText': {
                    'location': {'index': 1},
                    'text': content
                }
            }
        ]
        docs_service.documents().batchUpdate(documentId=doc_id, body={'requests': requests}).execute()

        return f"Document created successfully! URL: https://docs.google.com/document/d/{doc_id}/edit"
    except Exception as e:
        return f"Error creating Google Doc: {str(e)}"

In [ ]:
agent_prompt = """You are FunderWonder, a helpful research assistant seeking funding opportunities.

Your Goal:
Help researchers find grants that match their needs using the Grants.gov API.

Tools available:
1. search_grants(keywords): Searches for grants. ALWAYS use this first when a user asks for funding.
2. get_grant_details(opportunity_id): Gets full details for a specific grant ID.
3. create_proposal_doc(title, content): Creates a new Google Doc. Use this to save drafted proposals.

Process:
- When a user asks for grants, call `search_grants`.
- MANDATORY: When summarizing the list, ALWAYS include the numeric ID for every grant (e.g., "ID: 360918").
- If the user likes a grant, offer to draft a proposal.
- Once a proposal is drafted in the chat, ask the user if they want to save it to their Google Docs.
- ONLY call `create_proposal_doc` if the user explicitly asks to save it.
"""

grant_agent = create_agent(
    model=llm,
    tools=[search_grants, get_grant_details, create_proposal_doc],
    system_prompt=agent_prompt
)

Test prompts:
